In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
import warnings
import os
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 50)

# Plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
%matplotlib inline

In [2]:
# Loading data
transaction_train = pd.read_csv('data/raw/train_transaction.csv')
identity_train = pd.read_csv('data/raw/train_identity.csv')

# Merging on TransactionID
train = transaction_train.merge(identity_train, on='TransactionID', how='left')

print(f"Data loaded: {train.shape}")
print(f"Transactions: {len(train):,}")
print(f"Features: {len(train.columns)}")
print(f"Fraud rate: {train['isFraud'].mean()*100:.2f}%")

Data loaded: (590540, 434)
Transactions: 590,540
Features: 434
Fraud rate: 3.50%


### Dropping Features with >90% missing values

In [3]:
# Calculating missing percentages
missing_pct = (train.isnull().sum() / len(train) * 100).sort_values(ascending=False)

# Features with >90% missing
high_missing = missing_pct[missing_pct > 90].index.tolist()

print(f"Features with >90% missing: {len(high_missing)}")
print(f"Sample features to drop: {high_missing[:10]}")

# Dropping these features
train_cleaned = train.drop(columns=high_missing)

print(f"\nDropped {len(high_missing)} features")
print(f"   Original: {train.shape[1]} features")
print(f"   After drop: {train_cleaned.shape[1]} features")

# Updated train
train = train_cleaned

Features with >90% missing: 12
Sample features to drop: ['id_24', 'id_25', 'id_07', 'id_08', 'id_21', 'id_26', 'id_27', 'id_23', 'id_22', 'dist2']

Dropped 12 features
   Original: 434 features
   After drop: 422 features


### Extracting time based features

In [4]:
# TransactionDT is in seconds
# Converting to hours and days
train['Transaction_hour'] = (train['TransactionDT'] // 3600) % 24
train['Transaction_day'] = train['TransactionDT'] // (3600 * 24)
train['Transaction_weekday'] = train['Transaction_day'] % 7

# Defining Time of day categories
def categorize_time(hour):
    if 6 <= hour < 12:
        return 'morning'
    elif 12 <= hour < 18:
        return 'afternoon'
    elif 18 <= hour < 21:
        return 'evening'
    else:
        return 'night'

train['Time_of_day'] = train['Transaction_hour'].apply(categorize_time)

# Business hours flag
train['Is_business_hours'] = ((train['Transaction_hour'] >= 9) & 
                               (train['Transaction_hour'] <= 17)).astype(int)

# Adding Weekend flag
train['Is_weekend'] = (train['Transaction_weekday'].isin([5, 6])).astype(int)

print(" Created time features:")
print(" • Transaction_hour (0-23)")
print(" • Transaction_day (days since start)")
print(" • Transaction_weekday (0-6)")
print(" • Time_of_day (morning/afternoon/evening/night)")
print(" • Is_business_hours (0/1)")
print(" • Is_weekend (0/1)")

# Preview
print("\nSample values:")
print(train[['TransactionDT', 'Transaction_hour', 'Transaction_day', 
             'Transaction_weekday', 'Time_of_day']].head())

 Created time features:
 • Transaction_hour (0-23)
 • Transaction_day (days since start)
 • Transaction_weekday (0-6)
 • Time_of_day (morning/afternoon/evening/night)
 • Is_business_hours (0/1)
 • Is_weekend (0/1)

Sample values:
   TransactionDT  Transaction_hour  Transaction_day  Transaction_weekday  \
0          86400                 0                1                    1   
1          86401                 0                1                    1   
2          86469                 0                1                    1   
3          86499                 0                1                    1   
4          86506                 0                1                    1   

  Time_of_day  
0       night  
1       night  
2       night  
3       night  
4       night  


### Processing Amount Based Features

In [18]:
# Log transformation (helps with skewed distribution)
train['TransactionAmt_log'] = np.log1p(train['TransactionAmt'])

# Decimal part (cents)
train['TransactionAmt_decimal'] = train['TransactionAmt'] - train['TransactionAmt'].astype(int)

# Round amount flag (transactions ending in .00)
train['Is_round_amount'] = (train['TransactionAmt_decimal'] == 0).astype(int)

# Creating amount bins
train['Amount_bin'] = pd.cut(train['TransactionAmt'], 
                              bins=[0, 50, 100, 200, 500, 1000, float('inf')],
                              labels=['0-50', '50-100', '100-200', '200-500', '500-1000', '1000+'])

print(" Created amount features:")
print(" - TransactionAmt_log (log transformation)")
print(" - TransactionAmt_decimal (cents)")
print(" - Is_round_amount (0/1)")
print(" - Amount_bin (categorical bins)")

# Check round amounts distribution
print(f"\nRound amounts: {train['Is_round_amount'].mean()*100:.1f}% of transactions")

 Created amount features:
 - TransactionAmt_log (log transformation)
 - TransactionAmt_decimal (cents)
 - Is_round_amount (0/1)
 - Amount_bin (categorical bins)

Round amounts: 51.6% of transactions


### Card Interaction Features (Model Learns better from features combinings multiple card columns)

In [19]:
# Combining card features
train['card1_card2'] = train['card1'].astype(str) + '_' + train['card2'].astype(str) #Card No. + Card Type combination
train['card1_card5'] = train['card1'].astype(str) + '_' + train['card5'].astype(str) #Card No. + Address combination
train['card3_card5'] = train['card3'].astype(str) + '_' + train['card5'].astype(str) #Card Category + Address combination

# Card issuer + Card Product type combination
if 'card4' in train.columns and 'card6' in train.columns:
    train['card4_card6'] = train['card4'].astype(str) + '_' + train['card6'].astype(str)

print(" Created card combination features:")
print(" - card1_card2")
print(" - card1_card5")
print(" - card3_card5")
print(" - card4_card6")

# Cardinality
print(f"\nCardinality:")
print(f"card1_card2: {train['card1_card2'].nunique():,} unique combinations")
print(f"card1_card5: {train['card1_card5'].nunique():,} unique combinations")

 Created card combination features:
 - card1_card2
 - card1_card5
 - card3_card5
 - card4_card6

Cardinality:
card1_card2: 14,520 unique combinations
card1_card5: 14,180 unique combinations


### Email Features

In [20]:
# Email match (purchaser and recipient same domain)
if 'P_emaildomain' in train.columns and 'R_emaildomain' in train.columns:
    train['Email_domain_match'] = (train['P_emaildomain'] == train['R_emaildomain']).astype(int)
    
    # Filling missing as separate category
    train['P_emaildomain'] = train['P_emaildomain'].fillna('missing')
    train['R_emaildomain'] = train['R_emaildomain'].fillna('missing')
    
    # Common email providers
    common_providers = ['gmail.com', 'yahoo.com', 'hotmail.com', 'outlook.com', 'icloud.com']
    
    train['P_email_is_common'] = train['P_emaildomain'].isin(common_providers).astype(int)
    train['R_email_is_common'] = train['R_emaildomain'].isin(common_providers).astype(int)
    
    print("Created email features:")
    print("- Email_domain_match (0/1)")
    print("- P_email_is_common (0/1)")
    print("- R_email_is_common (0/1)")
    
    print(f"\nEmail domain matches: {train['Email_domain_match'].mean()*100:.1f}%")

Created email features:
- Email_domain_match (0/1)
- P_email_is_common (0/1)
- R_email_is_common (0/1)

Email domain matches: 31.5%


### Address match features

In [21]:
# Checking if purchaser and recipient addresses match
if 'addr1' in train.columns and 'addr2' in train.columns:
    train['addr_match'] = (train['addr1'] == train['addr2']).astype(int)
    print(" Created addr_match feature")
    print(f"Address matches: {train['addr_match'].mean()*100:.1f}%")
else:
    print("Address columns not found")

 Created addr_match feature
Address matches: 0.0%


### Handling D-Columns (Timedelta)

In [22]:
# D-columns represent time deltas
d_columns = [col for col in train.columns if col.startswith('D')]

print(f"Found {len(d_columns)} D-columns")

# Fill missing values with -1 (indicating "not available")
for col in d_columns:
    train[col] = train[col].fillna(-1)
    
print(f"Filled missing values in {len(d_columns)} D-columns with -1")

Found 16 D-columns
Filled missing values in 16 D-columns with -1


### Handling C-Columns

In [23]:
# C-columns represent counts
c_columns = [col for col in train.columns if col.startswith('C')]

print(f"Found {len(c_columns)} C-columns")

# Fill missing with 0 (no count)
for col in c_columns:
    train[col] = train[col].fillna(0)
    
print(f"Filled missing values in {len(c_columns)} C-columns with 0")

# Sum of all counts (activity indicator)
if len(c_columns) > 0:
    train['C_sum'] = train[c_columns].sum(axis=1)
    print("Created C_sum (total count across all C-columns)")

Found 15 C-columns
Filled missing values in 15 C-columns with 0
Created C_sum (total count across all C-columns)


### Handling V-Columns (Vesta features)

In [24]:
# Selecting top V-columns based on correlation (from EDA)
numerical_features = train.select_dtypes(include=[np.number]).columns.tolist()
numerical_features.remove('isFraud')
numerical_features.remove('TransactionID')

sample_size = min(100000, len(train))
train_sample = train[[col for col in train.columns if col.startswith('V')] + ['isFraud']].sample(sample_size, random_state=42)

correlations = train_sample.corr()['isFraud'].drop('isFraud')
correlations = correlations.abs().sort_values(ascending=False).head(35)

print(f"Selected top {len(correlations)} V-columns for modeling")

Selected top 35 V-columns for modeling


In [25]:
# V-columns are Vesta's engineered features (anonymized)
v_columns = [col for col in train.columns if col.startswith('V')]

print(f"Found {len(v_columns)} V-columns")

# Fill missing with median (safer than mean for these features)
for col in v_columns:
    median_val = train[col].median()
    train[col] = train[col].fillna(median_val)
    
print(f"Filled missing values in {len(v_columns)} V-columns with median")

Found 339 V-columns
Filled missing values in 339 V-columns with median


### Aggregating Usage Velocity Features

In [26]:
# Sorting by time
train_sorted = train.sort_values('TransactionDT').reset_index(drop=True)

# Grouping by card1 (user identifier)
print("Calculating transaction velocity by card1...")

# Counting transactions per card
card_counts = train_sorted.groupby('card1').size().to_dict()
train_sorted['card1_transaction_count'] = train_sorted['card1'].map(card_counts)

# Transaction frequency features
# For each card, calculating time since previous transaction
train_sorted['prev_TransactionDT'] = train_sorted.groupby('card1')['TransactionDT'].shift(1)
train_sorted['time_since_last_transaction'] = train_sorted['TransactionDT'] - train_sorted['prev_TransactionDT']

# Filling first transaction per card with a large number
train_sorted['time_since_last_transaction'] = train_sorted['time_since_last_transaction'].fillna(999999)

# Average transaction amount per card
card_avg_amt = train_sorted.groupby('card1')['TransactionAmt'].mean().to_dict()
train_sorted['card1_avg_amount'] = train_sorted['card1'].map(card_avg_amt)

# Deviation from user's typical amount
train_sorted['amount_deviation'] = train_sorted['TransactionAmt'] - train_sorted['card1_avg_amount']
train_sorted['amount_deviation_ratio'] = train_sorted['TransactionAmt'] / (train_sorted['card1_avg_amount'] + 1)

print(" Created velocity features:")
print(" - card1_transaction_count (total transactions per card)")
print(" - time_since_last_transaction (seconds since last transaction)")
print(" - card1_avg_amount (average transaction amount per card)")
print(" - amount_deviation (difference from user's average)")
print(" - amount_deviation_ratio (ratio to user's average)")

# Updated train
train = train_sorted

# Showing statistics
print(f"\nVelocity statistics:")
print(f" Max transactions per card: {train['card1_transaction_count'].max()}")
print(f" Avg time between transactions: {train['time_since_last_transaction'].median()/3600:.1f} hours")

Calculating transaction velocity by card1...
 Created velocity features:
 - card1_transaction_count (total transactions per card)
 - time_since_last_transaction (seconds since last transaction)
 - card1_avg_amount (average transaction amount per card)
 - amount_deviation (difference from user's average)
 - amount_deviation_ratio (ratio to user's average)

Velocity statistics:
 Max transactions per card: 14932
 Avg time between transactions: 1.8 hours


### Target Encoding (with smoothing) for High Cardinality Categorical columns

In [27]:
def target_encode(train_df, col, target='isFraud', smoothing=10):
    # Calculating global mean
    global_mean = train_df[target].mean()
    
    # Calculating encoding
    agg = train_df.groupby(col)[target].agg(['sum', 'count'])
    counts = agg['count']
    means = agg['sum'] / counts
    
    # Smoothing the encoding
    smooth = (counts * means + smoothing * global_mean) / (counts + smoothing)
    
    return train_df[col].map(smooth.to_dict()).fillna(global_mean)

# Features to target encode
high_cardinality_cols = ['card1', 'card2', 'card3', 'card5', 
                         'card1_card2', 'card1_card5', 'card3_card5',
                         'P_emaildomain', 'R_emaildomain']

for col in high_cardinality_cols:
    if col in train.columns:
        train[f'{col}_target_enc'] = target_encode(train, col)
        print(f"Target encoded: {col}")

print(f"\nCreated target-encoded features for {len(high_cardinality_cols)} columns")

Target encoded: card1
Target encoded: card2
Target encoded: card3
Target encoded: card5
Target encoded: card1_card2
Target encoded: card1_card5
Target encoded: card3_card5
Target encoded: P_emaildomain
Target encoded: R_emaildomain

Created target-encoded features for 9 columns


### Label Encoding for Low-Cardinality Categories

In [28]:
# For features with a few unique values - using label encoding
label_encode_cols = ['ProductCD', 'card4', 'card6', 'M1', 'M2', 'M3', 
                     'M4', 'M5', 'M6', 'M7', 'M8', 'M9',
                     'Time_of_day']

le = LabelEncoder()

for col in label_encode_cols:
    if col in train.columns:
        # Converting to string first, handle missing
        train[col] = train[col].fillna('missing').astype(str)
        train[f'{col}_encoded'] = le.fit_transform(train[col])
        print(f"Label encoded: {col} ({train[col].nunique()} categories)")

print(f"\nLabel encoded {len([c for c in label_encode_cols if c in train.columns])} features")

Label encoded: ProductCD (5 categories)
Label encoded: card4 (5 categories)
Label encoded: card6 (5 categories)
Label encoded: M1 (3 categories)
Label encoded: M2 (3 categories)
Label encoded: M3 (3 categories)
Label encoded: M4 (4 categories)
Label encoded: M5 (3 categories)
Label encoded: M6 (3 categories)
Label encoded: M7 (3 categories)
Label encoded: M8 (3 categories)
Label encoded: M9 (3 categories)
Label encoded: Time_of_day (4 categories)

Label encoded 13 features


### Final Imputation of Remaining Missing Values

In [29]:
# Checking remaining missing values
remaining_missing = train.isnull().sum().sum()
print(f"Remaining missing values: {remaining_missing:,}")

if remaining_missing > 0:
    # Numerical columns: filling with median
    numerical_cols = train.select_dtypes(include=[np.number]).columns
    for col in numerical_cols:
        if train[col].isnull().sum() > 0:
            median_val = train[col].median()
            train[col] = train[col].fillna(median_val)
    
    # Categorical columns: filling with 'missing'
    categorical_cols = train.select_dtypes(include=['object']).columns
    for col in categorical_cols:
        if train[col].isnull().sum() > 0:
            train[col] = train[col].fillna('missing')
    
    print("Filled all remaining missing values")
    print(f"Numerical features: filled with median")
    print(f"Categorical features: filled with 'missing'")
else:
    print("No missing values remaining!")

Remaining missing values: 13,553
Filled all remaining missing values
Numerical features: filled with median
Categorical features: filled with 'missing'


### Selecting Final Feature Set

In [30]:
# Identifying feature columns (exclude ID and target)
exclude_cols = ['TransactionID', 'isFraud', 'TransactionDT']

# Excluding original categorical columns (we have encoded versions)
original_categoricals = ['ProductCD', 'card4', 'card6', 'M1', 'M2', 'M3', 
                         'M4', 'M5', 'M6', 'M7', 'M8', 'M9',
                         'P_emaildomain', 'R_emaildomain', 
                         'card1_card2', 'card1_card5', 'card3_card5',
                         'Time_of_day']

# Excluding intermediate features
intermediate_cols = ['prev_TransactionDT']

# Consolidating final feature list
all_features = [col for col in train.columns 
                if col not in exclude_cols + original_categoricals + intermediate_cols
                and train[col].dtype in [np.int64, np.float64]]

print(f"Total features available: {len(all_features)}")

# Selecting only numerical features for modeling
feature_cols = all_features

print(f"Selected features for modeling: {len(feature_cols)}")

Total features available: 430
Selected features for modeling: 430


### Creating Time-Based Spilts (Train/Test/Validation)

In [31]:
# Sorting by transaction time
train_sorted = train.sort_values('TransactionDT').reset_index(drop=True)

# Calculating split indices
n = len(train_sorted)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

# Splitting data
train_set = train_sorted.iloc[:train_end].copy()
val_set = train_sorted.iloc[train_end:val_end].copy()
test_set = train_sorted.iloc[val_end:].copy()

print(f"Split sizes:")
print(f"Train: {len(train_set):,} ({len(train_set)/n*100:.1f}%)")
print(f"Validation: {len(val_set):,} ({len(val_set)/n*100:.1f}%)")
print(f"Test: {len(test_set):,} ({len(test_set)/n*100:.1f}%)")

# Verifying fraud rates
print(f"\nFraud rates:")
print(f"Train: {train_set['isFraud'].mean()*100:.2f}%")
print(f"Validation: {val_set['isFraud'].mean()*100:.2f}%")
print(f"Test: {test_set['isFraud'].mean()*100:.2f}%")

# Verifying time ranges don't overlap
print(f"\nTime ranges (non-overlapping):")
print(f"Train: {train_set['TransactionDT'].min()} to {train_set['TransactionDT'].max()}")
print(f"Val: {val_set['TransactionDT'].min()} to {val_set['TransactionDT'].max()}")
print(f"Test: {test_set['TransactionDT'].min()} to {test_set['TransactionDT'].max()}")

Split sizes:
Train: 413,378 (70.0%)
Validation: 88,581 (15.0%)
Test: 88,581 (15.0%)

Fraud rates:
Train: 3.52%
Validation: 3.43%
Test: 3.48%

Time ranges (non-overlapping):
Train: 86400 to 10437996
Val: 10438003 to 13151840
Test: 13151880 to 15811131


### Saving Processed Features

In [32]:
# Preparing final datasets
X_train = train_set[feature_cols]
y_train = train_set['isFraud']

X_val = val_set[feature_cols]
y_val = val_set['isFraud']

X_test = test_set[feature_cols]
y_test = test_set['isFraud']

# Saving to CSV
X_train.to_csv('data/processed/X_train.csv', index=False)
y_train.to_csv('data/processed/y_train.csv', index=False)

X_val.to_csv('data/processed/X_val.csv', index=False)
y_val.to_csv('data/processed/y_val.csv', index=False)

X_test.to_csv('data/processed/X_test.csv', index=False)
y_test.to_csv('data/processed/y_test.csv', index=False)

# Also saving feature names
with open('data/processed/feature_names.txt', 'w') as f:
    for col in feature_cols:
        f.write(f"{col}\n")

print("Saved processed datasets:")
print(f"X_train.csv: {X_train.shape}")
print(f"y_train.csv: {y_train.shape}")
print(f"X_val.csv: {X_val.shape}")
print(f"y_val.csv: {y_val.shape}")
print(f"X_test.csv: {X_test.shape}")
print(f"y_test.csv: {y_test.shape}")
print(f"feature_names.txt: {len(feature_cols)} features")

Saved processed datasets:
X_train.csv: (413378, 430)
y_train.csv: (413378,)
X_val.csv: (88581, 430)
y_val.csv: (88581,)
X_test.csv: (88581, 430)
y_test.csv: (88581,)
feature_names.txt: 430 features


### Summarizing Steps

In [35]:
print("\nCOMPLETED STEPS:")
print("1. Dropped features with >90% missing values")
print("2. Created time-based features (hour, day, weekend, etc.)")
print("3. Created amount-based features (log, decimal, bins)")
print("4. Created card combination features")
print("5. Created email domain features")
print("6. Created velocity features (transaction patterns)")
print("7. Created address match features")
print("8. Processed D-columns (timedelta features)")
print("9. Processed C-columns (count features)")
print("10. Processed V-columns (Vesta features)")
print("11. Target-encoded high-cardinality categories")
print("12. Label-encoded low-cardinality categories")
print("13. Created missing value indicators")
print("14. Imputed all remaining missing values")
print("15. Created time-based train/val/test splits")

print(f"\nKEY FEATURES CREATED:")
print(f"- Time features: 6")
print(f"- Amount features: 4")
print(f"- Card features: 4")
print(f"- Email features: 3")
print(f"- Velocity features: 5")
print(f"- Target-encoded features: {len([c for c in feature_cols if 'target_enc' in c])}")
print(f"- Label-encoded features: {len([c for c in feature_cols if 'encoded' in c])}")

print("Next step: Train baseline & advanced models -> Hyperparameter tuning -> Evaluation")


COMPLETED STEPS:
1. Dropped features with >90% missing values
2. Created time-based features (hour, day, weekend, etc.)
3. Created amount-based features (log, decimal, bins)
4. Created card combination features
5. Created email domain features
6. Created velocity features (transaction patterns)
7. Created address match features
8. Processed D-columns (timedelta features)
9. Processed C-columns (count features)
10. Processed V-columns (Vesta features)
11. Target-encoded high-cardinality categories
12. Label-encoded low-cardinality categories
13. Created missing value indicators
14. Imputed all remaining missing values
15. Created time-based train/val/test splits

KEY FEATURES CREATED:
- Time features: 6
- Amount features: 4
- Card features: 4
- Email features: 3
- Velocity features: 5
- Target-encoded features: 9
- Label-encoded features: 13
Next step: Train baseline & advanced models -> Hyperparameter tuning -> Evaluation
